<a href="https://colab.research.google.com/github/fisicorj/Astrophysics/blob/main/Estrelas_OB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

# ==========================
# CONFIGURAÇÕES DO USUÁRIO
# ==========================
DATA_FILE = "dados.xlsx"
ISO_FILE = "isocronas.xlsx"
SHEET_ESTRELAS = "berniniperonxshoot"
SHEET_ISOCRONA = "tracks2"
OUTPUT_PDF = "HR_B_diagrama.pdf"
NAME_GRAPHIC = "Diagrama HR - Estrelas Tipo B"
PLOT_ISOCRONA = True # Nova configuração para plotar ou não a isócrona
# ==========================

# Estilo gráfico
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 100

# Parâmetros de Plotagem Centralizados
PLOT_SETTINGS = {
    "fontsize_labels": 12,
    "fontsize_title": 14,
    "fontsize_text_stars": 7,
    "cmap_scatter": 'plasma',
    "s_scatter": 80,
    "alpha_scatter": 0.9,
    "color_isochrone": "gray",
    "linewidth_isochrone": 2.0,
    "alpha_isochrone": 0.8,
    "xlim_padding_teff": 1000,
    "ylim_padding_logl": 0.1,
    "text_offset_logl": 0.03,
    "stats_box_color": 'lightyellow',
    "stats_box_edgecolor": 'gray',
}

def load_and_clean_data(file_path: str, sheet_name: str) -> pd.DataFrame:
    """
    Carrega os dados das estrelas de um arquivo Excel e os pré-processa.

    Args:
        file_path (str): Caminho para o arquivo Excel.
        sheet_name (str): Nome da aba que contém os dados das estrelas.

    Returns:
        pd.DataFrame: DataFrame limpo com as colunas 'Star', 'Teff_K', 'logL', 'logg'.
                      Retorna um DataFrame vazio se o arquivo ou a sheet não forem encontrados,
                      ou se colunas essenciais estiverem faltando.
    """
    if not Path(file_path).exists():
        print(f"Erro: Arquivo de dados '{file_path}' não encontrado.")
        return pd.DataFrame()

    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
    except ValueError as e:
        print(f"Erro ao carregar a sheet '{sheet_name}' do arquivo '{file_path}': {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Erro inesperado ao ler o arquivo de dados '{file_path}': {e}")
        return pd.DataFrame()

    df.columns = df.columns.str.strip()
    required_cols = ["Teff(kK)", "log L/L0", "log g", "Star"]
    if not all(col in df.columns for col in required_cols):
        print(f"Erro: Uma ou mais colunas essenciais ({', '.join(required_cols)}) não encontradas na sheet '{sheet_name}'.")
        return pd.DataFrame()

    df["Teff_K"] = pd.to_numeric(df["Teff(kK)"], errors='coerce') * 1000
    df["logL"] = pd.to_numeric(df["log L/L0"], errors='coerce')
    df["logg"] = pd.to_numeric(df["log g"], errors='coerce')
    return df[["Star", "Teff_K", "logL", "logg"]].dropna()

def load_isochrone(file_path: str, sheet_name: str) -> pd.DataFrame:
    """
    Carrega os dados da isócrona de um arquivo Excel.

    Args:
        file_path (str): Caminho para o arquivo Excel.
        sheet_name (str): Nome da aba que contém os dados da isócrona.

    Returns:
        pd.DataFrame: DataFrame com os dados da isócrona, renomeando 'log_L' para 'logL'.
                      Retorna um DataFrame vazio se o arquivo ou a sheet não forem encontrados.
    """
    if not Path(file_path).exists():
        print(f"Erro: Arquivo de isócronas '{file_path}' não encontrado.")
        return pd.DataFrame()

    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name).dropna()
    except ValueError as e:
        print(f"Erro ao carregar a sheet '{sheet_name}' do arquivo '{file_path}': {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Erro inesperado ao ler o arquivo de isócronas '{file_path}': {e}")
        return pd.DataFrame()

    if "log_L" in df.columns:
        df.rename(columns={"log_L": "logL"}, inplace=True)
    elif "logL" not in df.columns:
        print(f"Erro: Colunas 'log_L' ou 'logL' não encontradas na sheet de isócrona '{sheet_name}'.")
        return pd.DataFrame()

    if "Teff" not in df.columns:
        print(f"Erro: Coluna 'Teff' não encontrada na sheet de isócrona '{sheet_name}'.")
        return pd.DataFrame()

    return df[["Teff", "logL"]]

def plot_hr_diagram(df: pd.DataFrame, iso: pd.DataFrame, pdf: PdfPages):
    """
    Gera o diagrama HR com as estrelas e a isócrona, e salva no PDF.

    Args:
        df (pd.DataFrame): DataFrame com os dados das estrelas.
        iso (pd.DataFrame): DataFrame com os dados da isócrona.
        pdf (PdfPages): Objeto PdfPages para salvar a figura.
    """
    if df.empty:
        print("Não há dados de estrelas para plotar o diagrama HR.")
        return

    fig, ax = plt.subplots(figsize=(14, 8))

    if PLOT_ISOCRONA and not iso.empty:
        ax.plot(iso["Teff"], iso["logL"],
                color=PLOT_SETTINGS["color_isochrone"],
                linewidth=PLOT_SETTINGS["linewidth_isochrone"],
                alpha=PLOT_SETTINGS["alpha_isochrone"],
                label="Isócrona")
        # Se você quiser que a legenda da isócrona apareça no gráfico:
        # ax.legend(loc='upper right')

    scatter = ax.scatter(df["Teff_K"], df["logL"], c=df["logg"], cmap=PLOT_SETTINGS["cmap_scatter"],
                         s=PLOT_SETTINGS["s_scatter"], alpha=PLOT_SETTINGS["alpha_scatter"],
                         edgecolors='black', linewidth=0.5, label="Estrelas")
    plt.colorbar(scatter, ax=ax, pad=0.01).set_label("log g (cgs)", fontsize=PLOT_SETTINGS["fontsize_labels"])

    for i, row in df.iterrows():
        ax.text(row["Teff_K"], row["logL"] + PLOT_SETTINGS["text_offset_logl"], row["Star"],
                fontsize=PLOT_SETTINGS["fontsize_text_stars"], ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6))

    ax.invert_xaxis()
    teff_min, teff_max = df["Teff_K"].min(), df["Teff_K"].max()
    logl_min, logl_max = df["logL"].min(), df["logL"].max()
    ax.set_xlim(teff_max + PLOT_SETTINGS["xlim_padding_teff"], teff_min - PLOT_SETTINGS["xlim_padding_teff"])
    ax.set_ylim(logl_min - PLOT_SETTINGS["ylim_padding_logl"], logl_max + PLOT_SETTINGS["ylim_padding_logl"])

    ax.set_xlabel("Temperatura Efetiva (K)", fontsize=PLOT_SETTINGS["fontsize_labels"], fontweight='bold')
    ax.set_ylabel(r"log(L / L$_{\odot}$)", fontsize=PLOT_SETTINGS["fontsize_labels"], fontweight='bold') # Uso de LaTeX
    ax.set_title(NAME_GRAPHIC, fontsize=PLOT_SETTINGS["fontsize_title"], fontweight='bold', pad=15)
    ax.grid(True, alpha=0.3, linestyle='--')

    ax.text(0.02, 0.02,
            f"{len(df)} estrelas | Teff média: {df['Teff_K'].mean():.0f} K | logL média: {df['logL'].mean():.2f}",
            transform=ax.transAxes, fontsize=9, ha='left', va='bottom',
            bbox=dict(facecolor=PLOT_SETTINGS["stats_box_color"],
                      edgecolor=PLOT_SETTINGS["stats_box_edgecolor"],
                      boxstyle='round,pad=0.4'))

    plt.tight_layout()
    pdf.savefig(fig)
    plt.close()

def plot_legend_page(df: pd.DataFrame, pdf: PdfPages):
    """
    Gera uma página de legenda com a tabela de estrelas e seus parâmetros.

    Args:
        df (pd.DataFrame): DataFrame com os dados das estrelas.
        pdf (PdfPages): Objeto PdfPages para salvar a figura.
    """
    if df.empty:
        print("Não há dados de estrelas para gerar a página de legenda.")
        return

    fig, ax = plt.subplots(figsize=(10, 12))
    ax.axis('off') # Desativa os eixos

    # Preparar os dados para a tabela
    table_data = []
    headers = ["Estrela", "Teff (K)", "log g"]
    for _, row in df.iterrows():
        table_data.append([row['Star'], int(row['Teff_K']), f"{row['logg']:.2f}"])

    # Criar a tabela
    table = ax.table(cellText=table_data,
                     colLabels=headers,
                     loc='center',
                     cellLoc='center',
                     colColours=['lightgray']*len(headers))

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.2) # Ajusta o tamanho da tabela

    ax.set_title("Tabela de Estrelas com Teff e log g", fontsize=PLOT_SETTINGS["fontsize_title"], fontweight='bold', pad=20)
    pdf.savefig(fig)
    plt.close()

def main():
    """Função principal para carregar dados, plotar e salvar o PDF."""
    print("Carregando dados das estrelas...")
    df = load_and_clean_data(DATA_FILE, SHEET_ESTRELAS)
    print("Carregando isócronas...")
    iso = load_isochrone(ISO_FILE, SHEET_ISOCRONA)

    if df.empty:
        print("Não foi possível processar os dados das estrelas. Verifique os arquivos de entrada.")
        return

    with PdfPages(OUTPUT_PDF) as pdf:
        print("Gerando Diagrama HR...")
        plot_hr_diagram(df, iso, pdf)
        print("Gerando página de legenda...")
        plot_legend_page(df, pdf)
    print(f"PDF '{OUTPUT_PDF}' gerado com sucesso!")

if __name__ == "__main__":
    main()

Carregando dados das estrelas...
Carregando isócronas...
Gerando Diagrama HR...
Gerando página de legenda...
PDF 'HR_B_diagrama.pdf' gerado com sucesso!
